# Reinforcement Learning Training of PMM Strategy Based on TD3

This notebook implements a PMM market making strategy training system based on the **TD3 (Twin Delayed Deep Deterministic Policy Gradient)** algorithm.

## TD3 Algorithm Features

- **Deterministic Policy**: Outputs deterministic actions, suitable for continuous control tasks
- **Dual Q Networks**: Uses two Critic networks to reduce Q-value overestimation
- **Delayed Policy Updates**: Reduces policy update frequency to improve stability
- **Target Policy Smoothing**: Adds noise to target actions to improve robustness
- **Clipped Double Q-Learning**: Uses smaller Q-values as targets to reduce overestimation

## Comparison with SAC

| Feature | TD3 | SAC |
|---------|-----|-----|
| Policy Type | Deterministic | Stochastic |
| Exploration | Add Noise | Entropy Regularization |
| Update Strategy | Delayed Updates | Updates Every Step |
| Target Smoothing | ✅ | ❌ |
| Computational Efficiency | Faster | Slower |

## 🚀 Quick Start

### Training Configuration (Important!)

Before running training, please set training parameters in **the first code cell**:
```python
NUM_EPISODES = 20    # Set training rounds directly: 20(test), 100(quick), 500(standard), 1000(deep)
BATCH_SIZE = 128     # Batch size (reduce if memory insufficient)
```

### Execution Order

1. **Cell 1**: Training configuration (set NUM_EPISODES and other parameters)
2. **Cell 2**: Environment setup and device selection
3. **Cell 3**: Data configuration and slice preparation
4. **Cell 4**: TD3 neural network components
5. **Cell 5**: TD3 algorithm implementation
6. **Cell 6**: Training configuration (automatically uses global parameters)
7. **Cell 7**: Environment manager
8. **Cell 8**: Training loop
9. **Cell 9**: Execute training
10. **Cell 10**: Model evaluation

### ⚠️ Notes

- **Quick Test**: Set `NUM_EPISODES=20` for 5-10 minute test
- **Standard Training**: Set `NUM_EPISODES=500` for stable strategy
- **Memory Issues**: Reduce `BATCH_SIZE` and `REPLAY_BUFFER_SIZE`
- **Save Progress**: Training periodically saves checkpoints for recovery

In [1]:
# ⚠️ Training Configuration (Set this before running other code)
# ====================================================

# Set training rounds directly (recommended approach)
NUM_EPISODES = 200         # Freely configurable: 20(test), 100(quick), 500(standard), 1000(deep)

# Other adjustable parameters
BATCH_SIZE = 128          # Batch size (reduce to 64 if memory insufficient)
REPLAY_BUFFER_SIZE = 20000  # Experience pool size (reduce to 10000 if memory insufficient)
DATA_SAMPLE_RATE = 0.1    # Data sampling rate (0.05-0.2)
SAVE_INTERVAL = 50        # Model save interval

# Display current configuration
print("=" * 50)
print("📋 TD3 Training Configuration")
print("=" * 50)

# Determine mode based on training episodes
if NUM_EPISODES <= 20:
    print("🧪 Mode: Quick Test")
    estimated_time = "5-10 minutes"
elif NUM_EPISODES <= 100:
    print("⚡ Mode: Fast Training")
    estimated_time = "30-60 minutes"
elif NUM_EPISODES <= 500:
    print("💪 Mode: Standard Training")
    estimated_time = "4-6 hours"
else:
    print("🔥 Mode: Deep Training")
    estimated_time = "8-12 hours"

print(f"📊 Training Episodes: {NUM_EPISODES:,} episodes")
print(f"💾 Batch Size: {BATCH_SIZE}")
print(f"🗄️ Experience Pool Size: {REPLAY_BUFFER_SIZE:,}")
print(f"📉 Data Sampling Rate: {DATA_SAMPLE_RATE*100:.0f}%")
print(f"💾 Save Interval: Every {SAVE_INTERVAL} episodes")
print(f"\n⏱️ Estimated Time: {estimated_time}")

print("\n💡 Recommended Configurations:")
print("   • Test Environment: NUM_EPISODES=20")
print("   • Quick Prototype: NUM_EPISODES=100")
print("   • Standard Training: NUM_EPISODES=500")
print("   • Best Performance: NUM_EPISODES=1000")
print("=" * 50)

📋 TD3 Training Configuration
💪 Mode: Standard Training
📊 Training Episodes: 200 episodes
💾 Batch Size: 128
🗄️ Experience Pool Size: 20,000
📉 Data Sampling Rate: 10%
💾 Save Interval: Every 50 episodes

⏱️ Estimated Time: 4-6 hours

💡 Recommended Configurations:
   • Test Environment: NUM_EPISODES=20
   • Quick Prototype: NUM_EPISODES=100
   • Standard Training: NUM_EPISODES=500
   • Best Performance: NUM_EPISODES=1000


In [2]:
# Cell 1: Environment setup and dependency imports
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import warnings
import os
from collections import deque
import random
from tensordict import TensorDict
from hftbacktest import BacktestAsset
from lib.rl_env import create_pmm_env
from lib.data_slicer import DataSlicer
from tqdm.notebook import tqdm
import json
import time
import copy

# Set warning filters and random seeds
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# 🔧 Global trading configuration
MAKER_FEE_RATE = -0.00003     # Maker fee rate -0.003% (negative fee rebate)
TAKER_FEE_RATE = 0.0007       # Taker fee rate +0.07% (positive fee charge)
TICK_SIZE = 0.0001            # XRP minimum price movement unit
LOT_SIZE = 0.1                # XRP minimum trading quantity unit
TRAINING_PAIR = 'xrpusdt'     # XRP trading pair
START_DATE = 20250717         # Use updated data

print(f"📊 Global Trading Configuration:")
print(f"   Trading Pair: {TRAINING_PAIR.upper()}")
print(f"   Maker Fee Rate: {MAKER_FEE_RATE*100:.4f}% (negative fee rebate)")
print(f"   Taker Fee Rate: {TAKER_FEE_RATE*100:.4f}% (positive fee charge)")
print(f"   Minimum Price Unit: {TICK_SIZE}")
print(f"   Minimum Trading Unit: {LOT_SIZE}")

# 🔧 Device configuration (smart selection: CUDA > MPS > CPU)
def select_device():
    """Intelligently select the best available device"""
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("🚀 Using CUDA GPU acceleration")
        print(f"   GPU Model: {torch.cuda.get_device_name(0)}")
        print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
        torch.cuda.empty_cache()
        return device
    
    if torch.backends.mps.is_available():
        try:
            test_tensor = torch.tensor([1.0], device="mps")
            _ = test_tensor * 2
            device = torch.device("mps")
            print("🍎 Using Apple Silicon MPS acceleration")
            print("   Tip: MPS acceleration may significantly improve training speed")
            return device
        except Exception as e:
            print(f"⚠️ MPS available but initialization failed: {e}")
            print("   Falling back to CPU mode")
    
    device = torch.device("cpu")
    print("💻 Using CPU")
    print("   Tip: Training speed is slower, recommend using GPU or MPS")
    import platform
    print(f"   Processor: {platform.processor()}")
    print(f"   CPU Cores: {os.cpu_count()}")
    return device

device = select_device()
print(f"✅ TD3 reinforcement learning environment initialization complete, using device: {device}")

📊 Global Trading Configuration:
   Trading Pair: XRPUSDT
   Maker Fee Rate: -0.0030% (negative fee rebate)
   Taker Fee Rate: 0.0700% (positive fee charge)
   Minimum Price Unit: 0.0001
   Minimum Trading Unit: 0.1
🍎 Using Apple Silicon MPS acceleration
   Tip: MPS acceleration may significantly improve training speed
✅ TD3 reinforcement learning environment initialization complete, using device: mps


In [3]:
# Cell 2: Data Configuration
from lib.data_slicer import DataSlicer
print("📊 Initializing data system...")

# Check data file
data_file = f'data/output/{TRAINING_PAIR}_{START_DATE}.npz'
if not os.path.exists(data_file):
    raise FileNotFoundError(f"Data file does not exist: {data_file}")

print(f"✅ Data file: {os.path.basename(data_file)}")

# Create data segments
print("📊 Preparing data segments (sliced by time)...")
slicer = DataSlicer(slices_dir='data/slices')
data_splits = slicer.split_data_by_time(
    data_file=data_file,
    pair_name=TRAINING_PAIR,
    start_date=START_DATE,
    hours_per_split=0.167  # 10 minutes per segment (10/60 hours)
)
print(f"✅ Obtained {len(data_splits)} data segments")

data_files = data_splits
print(f"✅ Data preparation complete, total {len(data_files)} training segments")

# Show information for first few slices
for i, split_file in enumerate(data_files[:3]):
    info = slicer.get_split_info(split_file)
    print(f"   Segment{i}: {info['records']:,} records, {info['duration_hours']:.1f} hours")

📊 Initializing data system...
✅ Data file: xrpusdt_20250717.npz
📊 Preparing data segments (sliced by time)...
📦 Found existing data slices (144 files), using directly...
   Segment 0: 349,899 records (0.3M) - 0.2 hours
   Segment 1: 330,220 records (0.3M) - 0.2 hours
   Segment 2: 343,814 records (0.3M) - 0.2 hours
   ... Total 144 segments
✅ Obtained 144 data segments
✅ Data preparation complete, total 144 training segments
   Segment0: 349,899 records, 0.2 hours
   Segment1: 330,220 records, 0.2 hours
   Segment2: 343,814 records, 0.2 hours


In [4]:
# Cell 3: TD3 Neural Network Components

class Actor(nn.Module):
    """TD3 Actor Network - deterministic policy"""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256, max_action=1.0):
        super(Actor, self).__init__()
        self.max_action = max_action
        
        # Network layers
        self.fc1 = nn.Linear(state_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, action_dim)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()
    
    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        action = torch.tanh(self.fc3(x))
        return action * self.max_action


class Critic(nn.Module):
    """TD3 Critic Network - twin Q-network architecture"""
    
    def __init__(self, state_dim, action_dim, hidden_dim=256):
        super(Critic, self).__init__()
        
        # Q1 Network
        self.q1_fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.q1_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q1_out = nn.Linear(hidden_dim, 1)
        
        # Q2 Network
        self.q2_fc1 = nn.Linear(state_dim + action_dim, hidden_dim)
        self.q2_fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.q2_out = nn.Linear(hidden_dim, 1)
        
        # Initialize weights
        self.apply(self._init_weights)
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                module.bias.data.zero_()
    
    def forward(self, state, action):
        xu = torch.cat([state, action], dim=1)
        
        # Q1 forward pass
        q1 = F.relu(self.q1_fc1(xu))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)
        
        # Q2 forward pass
        q2 = F.relu(self.q2_fc1(xu))
        q2 = F.relu(self.q2_fc2(q2))
        q2 = self.q2_out(q2)
        
        return q1, q2
    
    def Q1(self, state, action):
        """Calculate only Q1 value"""
        xu = torch.cat([state, action], dim=1)
        q1 = F.relu(self.q1_fc1(xu))
        q1 = F.relu(self.q1_fc2(q1))
        q1 = self.q1_out(q1)
        return q1


class ReplayBuffer:
    """Experience replay buffer"""
    
    def __init__(self, capacity=1000000):
        self.capacity = capacity
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        """Add experience to buffer"""
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        """Randomly sample a batch of experiences"""
        batch = random.sample(self.buffer, batch_size)
        state, action, reward, next_state, done = map(np.stack, zip(*batch))
        return state, action, reward, next_state, done
    
    def __len__(self):
        return len(self.buffer)


print("✅ TD3 neural network components definition complete")
print(f"   - Actor Network: deterministic policy network")
print(f"   - Critic Network: twin Q-network architecture")
print(f"   - ReplayBuffer: experience replay buffer")

✅ TD3 neural network components definition complete
   - Actor Network: deterministic policy network
   - Critic Network: twin Q-network architecture
   - ReplayBuffer: experience replay buffer


In [5]:
# Cell 4: TD3 Algorithm Implementation

class TD3:
    """Twin Delayed Deep Deterministic Policy Gradient algorithm implementation"""
    
    def __init__(
        self,
        state_dim,
        action_dim,
        action_low,
        action_high,
        device,
        lr_actor=3e-4,
        lr_critic=3e-4,
        gamma=0.99,
        tau=0.005,
        policy_noise=0.2,
        noise_clip=0.5,
        policy_freq=2,
        exploration_noise=0.1
    ):
        self.device = device
        self.gamma = gamma
        self.tau = tau
        self.policy_noise = policy_noise
        self.noise_clip = noise_clip
        self.policy_freq = policy_freq
        self.exploration_noise = exploration_noise
        
        # Action space bounds
        self.action_low = torch.tensor(action_low, device=device)
        self.action_high = torch.tensor(action_high, device=device)
        self.action_scale = (self.action_high - self.action_low) / 2.0
        self.action_bias = (self.action_high + self.action_low) / 2.0
        
        # Create networks
        self.actor = Actor(state_dim, action_dim).to(device)
        self.actor_target = Actor(state_dim, action_dim).to(device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        
        self.critic = Critic(state_dim, action_dim).to(device)
        self.critic_target = Critic(state_dim, action_dim).to(device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        
        # Optimizers
        self.actor_optimizer = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # Counter
        self.total_it = 0
    
    def select_action(self, state, add_noise=True):
        """Select action"""
        state = torch.FloatTensor(state).to(self.device).unsqueeze(0)
        
        with torch.no_grad():
            action = self.actor(state).squeeze(0).cpu().numpy()
        
        # Add exploration noise
        if add_noise:
            noise = np.random.normal(0, self.exploration_noise, size=action.shape)
            action = action + noise
        
        # Scale action from [-1, 1] to actual range
        action = action * self.action_scale.cpu().numpy() + self.action_bias.cpu().numpy()
        
        # Ensure actions are within bounds and integers for discrete parameters
        action = np.clip(action, self.action_low.cpu().numpy(), self.action_high.cpu().numpy())
        action[0] = round(action[0])  # half_spread
        action[1] = round(action[1])  # skew
        action[2] = round(action[2])  # grid_num
        action[3] = round(action[3])  # grid_interval
        
        return action
    
    def update(self, replay_buffer, batch_size=256):
        """Update network parameters"""
        self.total_it += 1
        
        if len(replay_buffer) < batch_size:
            return {}
        
        # Sample from experience buffer
        state, action, reward, next_state, done = replay_buffer.sample(batch_size)
        
        state = torch.FloatTensor(state).to(self.device)
        next_state = torch.FloatTensor(next_state).to(self.device)
        action = torch.FloatTensor(action).to(self.device)
        reward = torch.FloatTensor(reward).to(self.device).unsqueeze(1)
        done = torch.FloatTensor(done).to(self.device).unsqueeze(1)
        
        # Normalize actions to [-1, 1]
        action_normalized = (action - self.action_bias) / self.action_scale
        
        with torch.no_grad():
            # Select next action and add noise (target policy smoothing)
            next_action = self.actor_target(next_state)
            noise = torch.randn_like(next_action) * self.policy_noise
            noise = noise.clamp(-self.noise_clip, self.noise_clip)
            next_action = (next_action + noise).clamp(-1, 1)
            
            # Calculate target Q value
            target_q1, target_q2 = self.critic_target(next_state, next_action)
            target_q = torch.min(target_q1, target_q2)
            target_q_value = reward + (1 - done) * self.gamma * target_q
        
        # Update Critic
        current_q1, current_q2 = self.critic(state, action_normalized)
        critic_loss = F.mse_loss(current_q1, target_q_value) + F.mse_loss(current_q2, target_q_value)
        
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()
        
        # Delayed policy updates
        actor_loss = None
        if self.total_it % self.policy_freq == 0:
            # Calculate Actor loss
            actor_loss = -self.critic.Q1(state, self.actor(state)).mean()
            
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()
            
            # Soft update target networks
            for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
            
            for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)
        
        return {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item() if actor_loss is not None else 0,
            'total_iterations': self.total_it
        }
    
    def save(self, filepath):
        """Save model"""
        torch.save({
            'actor_state_dict': self.actor.state_dict(),
            'actor_target_state_dict': self.actor_target.state_dict(),
            'critic_state_dict': self.critic.state_dict(),
            'critic_target_state_dict': self.critic_target.state_dict(),
            'actor_optimizer_state_dict': self.actor_optimizer.state_dict(),
            'critic_optimizer_state_dict': self.critic_optimizer.state_dict(),
            'total_it': self.total_it
        }, filepath)
    
    def load(self, filepath):
        """Load model"""
        checkpoint = torch.load(filepath, map_location=self.device)
        self.actor.load_state_dict(checkpoint['actor_state_dict'])
        self.actor_target.load_state_dict(checkpoint['actor_target_state_dict'])
        self.critic.load_state_dict(checkpoint['critic_state_dict'])
        self.critic_target.load_state_dict(checkpoint['critic_target_state_dict'])
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer_state_dict'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer_state_dict'])
        self.total_it = checkpoint['total_it']


print("✅ TD3 algorithm implementation complete")
print(f"   - Deterministic policy: no entropy regularization")
print(f"   - Delayed updates: policy updated every {2} steps")
print(f"   - Target smoothing: reduces target Q-value variance")
print(f"   - Twin Q-networks: reduces Q-value overestimation")

✅ TD3 algorithm implementation complete
   - Deterministic policy: no entropy regularization
   - Delayed updates: policy updated every 2 steps
   - Target smoothing: reduces target Q-value variance
   - Twin Q-networks: reduces Q-value overestimation


In [6]:
# Cell 5: Training Configuration

TD3_CONFIG = {
    # Environment parameters
    'state_dim': 4,                    # Observation dimension
    'action_dim': 4,                   # Action dimension
    'action_low': [1.0, 1.0, 5.0, 1.0],    # Action lower bounds
    'action_high': [20.0, 30.0, 10.0, 20.0],  # Action upper bounds
    
    # TD3 hyperparameters
    'lr_actor': 3e-4,                  # Actor learning rate
    'lr_critic': 3e-4,                 # Critic learning rate
    'gamma': 0.99,                     # Discount factor
    'tau': 0.005,                      # Soft update coefficient
    'policy_noise': 0.2,               # Target policy noise
    'noise_clip': 0.5,                 # Noise clipping
    'policy_freq': 2,                  # Policy update frequency
    'exploration_noise': 0.1,          # Exploration noise
    
    # Training parameters (using global configuration)
    'batch_size': BATCH_SIZE,                 # Batch size
    'replay_buffer_size': REPLAY_BUFFER_SIZE, # Experience buffer size
    'num_episodes': NUM_EPISODES,             # Total training episodes
    'start_steps': 500,                      # Random exploration steps
    'update_interval': 1,                     # Update interval
    'eval_interval': 50,                      # Evaluation interval
    'save_interval': SAVE_INTERVAL,           # Save interval
    
    # Environment parameters
    'step_interval_ns': 2_500_000_000,  # Step interval (1 second)
    'max_steps_per_episode': 500,       # Maximum steps per episode
    
    # Data management (using global configuration)
    'use_data_slices': True,                  # Use data slices
    'hours_per_slice': 0.167,                 # Hours per slice (10 minutes)
    'slices_per_episode': 1,                  # Slices per episode
    'data_sample_rate': DATA_SAMPLE_RATE,     # Data sampling rate
    'max_samples_per_env': 5000000,           # Maximum sample limit
}

# Create model save directories
os.makedirs('checkpoints/td3', exist_ok=True)
os.makedirs('logs', exist_ok=True)

print("✅ TD3 training configuration complete (using global configuration)")
print(f"   State dimension: {TD3_CONFIG['state_dim']}")
print(f"   Action dimension: {TD3_CONFIG['action_dim']}")
print(f"   Learning rates: Actor={TD3_CONFIG['lr_actor']}, Critic={TD3_CONFIG['lr_critic']}")
print(f"   Batch size: {TD3_CONFIG['batch_size']} (global configuration)")
print(f"   Experience buffer capacity: {TD3_CONFIG['replay_buffer_size']:,} (global configuration)")
print(f"   Training episodes: {TD3_CONFIG['num_episodes']:,} (global configuration)")
print(f"   Policy update frequency: Every {TD3_CONFIG['policy_freq']} steps")
print(f"   Exploration noise: {TD3_CONFIG['exploration_noise']}")

✅ TD3 training configuration complete (using global configuration)
   State dimension: 4
   Action dimension: 4
   Learning rates: Actor=0.0003, Critic=0.0003
   Batch size: 128 (global configuration)
   Experience buffer capacity: 20,000 (global configuration)
   Training episodes: 200 (global configuration)
   Policy update frequency: Every 2 steps
   Exploration noise: 0.1


In [7]:
# Cell 6: Environment Manager

class EnvironmentManager:
    """Manage training environments for multiple data slices"""
    
    def __init__(self, data_files, config, device):
        self.data_files = data_files
        self.config = config
        self.device = device
        self.current_idx = 0
        self.slicer = DataSlicer()
        self.sample_rate = config.get('data_sample_rate', 0.1)
        self.max_samples = config.get('max_samples_per_env', 500000)
        print(f"   Environment manager using device: {self.device}")
        print(f"   Data sampling rate: {self.sample_rate*100:.0f}%")
        print(f"   Maximum samples: {self.max_samples:,}")
    
    def create_env_from_slice(self, data_file, sample_rate=None):
        """Create environment from data slice"""
        import gc
        
        if sample_rate is None:
            sample_rate = self.sample_rate
        
        # Clear memory
        gc.collect()
        if self.device.type == 'cuda':
            torch.cuda.empty_cache()
        elif self.device.type == 'mps':
            torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
        
        # Load data (silent mode)
        data = np.load(data_file)
        data_array = data['data']
        
        original_size = len(data_array)
        
        # Calculate sample quantity
        target_sample_size = int(original_size * sample_rate)
        actual_sample_size = min(target_sample_size, self.max_samples)
        
        # Sample data
        if actual_sample_size < original_size:
            indices = np.sort(np.random.choice(
                original_size, actual_sample_size, replace=False))
            data_array = data_array[indices]
        
        data.close()
        del data
        gc.collect()
        
        # Create HFT backtest asset
        data_asset = (
            BacktestAsset()
            .data([data_array])
            .linear_asset(1.0)
            .risk_adverse_queue_model()
            .no_partial_fill_exchange()
            .constant_latency(10_000_000, 10_000_000)
            .tick_size(TICK_SIZE)
            .lot_size(LOT_SIZE)
            .trading_value_fee_model(MAKER_FEE_RATE, TAKER_FEE_RATE)
            .power_prob_queue_model(3.0)
        )
        
        # Create PMM environment
        env = create_pmm_env(
            data_asset=data_asset,
            action_low=self.config['action_low'],
            action_high=self.config['action_high'],
            max_steps=self.config['max_steps_per_episode'],
            device=str(self.device),
            risk_penalty_weight=0.01,
            step_interval_ns=self.config['step_interval_ns'],
        )
        
        return env
    
    def get_next_env(self):
        """Get next training environment"""
        data_file = self.data_files[self.current_idx]
        env = self.create_env_from_slice(data_file)
        self.current_idx = (self.current_idx + 1) % len(self.data_files)
        return env, data_file
    
    def get_random_env(self):
        """Randomly get a training environment"""
        data_file = random.choice(self.data_files)
        self.current_idx = self.data_files.index(data_file)
        env = self.create_env_from_slice(data_file)
        return env, data_file


# Create environment manager
data_slices = data_files
print(f"✅ Using {len(data_slices)} data slices")
print(f"   Duration per slice: {TD3_CONFIG['hours_per_slice']} hours")
print(f"   Data sampling rate: {TD3_CONFIG.get('data_sample_rate', 0.1)*100:.0f}%")
print(f"   Maximum samples: {TD3_CONFIG.get('max_samples_per_env', 500000):,}")

env_manager = EnvironmentManager(data_slices, TD3_CONFIG, device)
print("✅ Environment manager created successfully")

✅ Using 144 data slices
   Duration per slice: 0.167 hours
   Data sampling rate: 10%
   Maximum samples: 5,000,000
   Environment manager using device: mps
   Data sampling rate: 10%
   Maximum samples: 5,000,000
✅ Environment manager created successfully


In [8]:
# Cell 7: TD3 Training Loop

def train_td3():
    """TD3 main training loop"""
    import gc
    
    # Initialize TD3 algorithm
    td3 = TD3(
        state_dim=TD3_CONFIG['state_dim'],
        action_dim=TD3_CONFIG['action_dim'],
        action_low=TD3_CONFIG['action_low'],
        action_high=TD3_CONFIG['action_high'],
        device=device,
        lr_actor=TD3_CONFIG['lr_actor'],
        lr_critic=TD3_CONFIG['lr_critic'],
        gamma=TD3_CONFIG['gamma'],
        tau=TD3_CONFIG['tau'],
        policy_noise=TD3_CONFIG['policy_noise'],
        noise_clip=TD3_CONFIG['noise_clip'],
        policy_freq=TD3_CONFIG['policy_freq'],
        exploration_noise=TD3_CONFIG['exploration_noise']
    )
    
    # Initialize experience replay buffer
    replay_buffer = ReplayBuffer(TD3_CONFIG['replay_buffer_size'])
    
    # Training statistics
    episode_rewards = []
    episode_steps = []
    training_losses = []
    total_steps = 0
    best_reward = -float('inf')
    
    print(f"\n🎯 Starting TD3 training")
    print(f"   Device: {device}")
    print(f"   Total episodes: {TD3_CONFIG['num_episodes']:,}")
    print(f"   Random exploration steps: {TD3_CONFIG['start_steps']:,}")
    print(f"   Batch size: {TD3_CONFIG['batch_size']}")
    
    # Training loop
    for episode in tqdm(range(TD3_CONFIG['num_episodes']), desc="Training Progress"):
        try:
            # Periodic memory cleanup
            if episode % 10 == 0:
                gc.collect()
                if device.type == 'cuda':
                    torch.cuda.empty_cache()
                elif device.type == 'mps':
                    torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
            
            # Get new environment
            if episode % 100 == 0:
                print(f"\n📌 Episode {episode+1}/{TD3_CONFIG['num_episodes']}")
            
            env, data_file = env_manager.get_next_env()
            
            # Reset environment
            tensordict = env.reset()
            state = tensordict['observation'].cpu().numpy()
            
            episode_reward = 0
            episode_step = 0
            
            # Episode loop
            done = False
            while not done and episode_step < TD3_CONFIG['max_steps_per_episode']:
                # Select action
                if total_steps < TD3_CONFIG['start_steps']:
                    # Random exploration
                    action = np.random.uniform(
                        TD3_CONFIG['action_low'],
                        TD3_CONFIG['action_high']
                    )
                    # Ensure integer parameters
                    action[0] = round(action[0])
                    action[1] = round(action[1])
                    action[2] = round(action[2])
                    action[3] = round(action[3])
                else:
                    # TD3 policy action selection
                    action = td3.select_action(state, add_noise=True)
                
                # Execute action
                action_tensor = torch.tensor(action, dtype=torch.float32, device=device)
                action_td = TensorDict({"action": action_tensor}, batch_size=(), device=device)
                
                try:
                    next_tensordict = env.step(action_td)
                except Exception as e:
                    if episode % 100 == 0:
                        print(f"   ⚠️ Step error: {e}")
                    done = True
                    break
                
                if 'next' in next_tensordict:
                    next_state = next_tensordict['next']['observation'].cpu().numpy()
                    reward = next_tensordict['next']['reward'].cpu().item()
                    done = next_tensordict['next']['done'].cpu().item() > 0.5
                    
                    # Store experience
                    replay_buffer.push(state, action, reward, next_state, done)
                    
                    # Update state
                    state = next_state
                    episode_reward += reward
                    episode_step += 1
                    total_steps += 1
                    
                    # Update networks
                    if total_steps >= TD3_CONFIG['start_steps'] and \
                       total_steps % TD3_CONFIG['update_interval'] == 0 and \
                       len(replay_buffer) >= TD3_CONFIG['batch_size']:
                        losses = td3.update(replay_buffer, TD3_CONFIG['batch_size'])
                        if losses:
                            training_losses.append(losses)
                else:
                    done = True
            
            # Record episode statistics
            episode_rewards.append(episode_reward)
            episode_steps.append(episode_step)
            
            # Update best reward
            if episode_reward > best_reward:
                best_reward = episode_reward
                best_model_path = "checkpoints/td3/td3_model_best.pth"
                td3.save(best_model_path)
            
            # Clean up environment
            env.close()
            del env
            
        except Exception as e:
            if episode % 100 == 0:
                print(f"   ❌ Episode {episode+1} error: {e}")
            continue
        
        # Periodic evaluation and saving
        if (episode + 1) % TD3_CONFIG['eval_interval'] == 0:
            if episode_rewards:
                recent_rewards = episode_rewards[-min(TD3_CONFIG['eval_interval'], len(episode_rewards)):]
                avg_reward = np.mean(recent_rewards)
                avg_steps = np.mean(episode_steps[-min(TD3_CONFIG['eval_interval'], len(episode_steps)):])
                
                print(f"\n📊 Episode {episode + 1}/{TD3_CONFIG['num_episodes']}")
                print(f"   Average reward: {avg_reward:.4f}")
                print(f"   Best reward: {best_reward:.4f}")
                print(f"   Average steps: {avg_steps:.0f}")
                print(f"   Total steps: {total_steps:,}")
                print(f"   Buffer size: {len(replay_buffer):,}")
                
                # Show recent losses
                if training_losses and len(training_losses) > 0:
                    recent_losses = training_losses[-min(100, len(training_losses)):]
                    avg_critic_loss = np.mean([l['critic_loss'] for l in recent_losses])
                    avg_actor_loss = np.mean([l['actor_loss'] for l in recent_losses if l['actor_loss'] > 0])
                    print(f"   Average Critic loss: {avg_critic_loss:.6f}")
                    if avg_actor_loss > 0:
                        print(f"   Average Actor loss: {avg_actor_loss:.6f}")
        
        # Periodic model saving
        if (episode + 1) % TD3_CONFIG['save_interval'] == 0:
            model_path = f"checkpoints/td3/td3_model_episode_{episode + 1}.pth"
            td3.save(model_path)
            print(f"💾 Model saved: {model_path}")
            
            # Save training progress
            progress = {
                'episode': episode + 1,
                'total_steps': total_steps,
                'best_reward': best_reward,
                'recent_rewards': episode_rewards[-100:] if len(episode_rewards) > 100 else episode_rewards,
            }
            progress_path = f"checkpoints/td3/training_progress.json"
            with open(progress_path, 'w') as f:
                json.dump(progress, f, indent=2)
    
    # Save final model
    if episode_rewards:
        final_model_path = "checkpoints/td3/td3_model_final.pth"
        td3.save(final_model_path)
        print(f"\n✅ Training complete! Final model saved: {final_model_path}")
        
        # Save complete training statistics
        stats = {
            'episode_rewards': episode_rewards,
            'episode_steps': episode_steps,
            'total_episodes': len(episode_rewards),
            'total_steps': total_steps,
            'best_reward': best_reward,
            'config': TD3_CONFIG
        }
        
        stats_path = f"logs/td3_training_stats_{time.strftime('%Y%m%d_%H%M%S')}.json"
        with open(stats_path, 'w') as f:
            json.dump(stats, f, indent=2)
        print(f"📊 Training statistics saved: {stats_path}")
    
    return td3, episode_rewards


print("✅ TD3 training system ready")
print(f"   - Using {len(data_slices)} data slices in rotation for training")
print(f"   - Data sampling {TD3_CONFIG['data_sample_rate']*100:.0f}%")
print(f"   - Supporting device acceleration: {device}")
print(f"   - Delayed policy updates: every {TD3_CONFIG['policy_freq']} steps")

✅ TD3 training system ready
   - Using 144 data slices in rotation for training
   - Data sampling 10%
   - Supporting device acceleration: mps
   - Delayed policy updates: every 2 steps


In [9]:
# Cell 9: Execute TD3 Training

print("🎯 Preparing to start TD3 training...")
print(f"   Device: {device}")
print(f"   Data slices: {len(data_slices)} slices")

print("\n📊 Current configuration (from global settings):")
print(f"   Training episodes: {TD3_CONFIG['num_episodes']:,}")
print(f"   Batch size: {TD3_CONFIG['batch_size']}")
print(f"   Experience buffer: {TD3_CONFIG['replay_buffer_size']:,}")
print(f"   Data sampling rate: {TD3_CONFIG['data_sample_rate']*100:.0f}%")
print(f"   Max steps per episode: {TD3_CONFIG['max_steps_per_episode']}")
print(f"   Save interval: Every {TD3_CONFIG['save_interval']} episodes")

# Show expectations based on training episodes
if TD3_CONFIG['num_episodes'] <= 20:
    print(f"\n🧪 Quick test mode: {TD3_CONFIG['num_episodes']} episodes")
    print("   Expected time: 5-10 minutes")
    print("   Purpose: Validate environment configuration")
elif TD3_CONFIG['num_episodes'] <= 100:
    print(f"\n⚡ Fast training mode: {TD3_CONFIG['num_episodes']} episodes")
    print("   Expected time: 30-60 minutes")
    print("   Purpose: Quick prototype validation")
elif TD3_CONFIG['num_episodes'] <= 500:
    print(f"\n💪 Standard training mode: {TD3_CONFIG['num_episodes']} episodes")
    print("   Expected time: 4-6 hours")
    print("   Purpose: Obtain usable strategy")
else:
    print(f"\n🔥 Deep training mode: {TD3_CONFIG['num_episodes']} episodes")
    print("   Expected time: 8-12 hours")
    print("   Purpose: Achieve best performance")
    print("   Suggestion: Let program run in background")

print("\n💡 Tip: Adjust NUM_EPISODES in the first configuration cell")

# Pre-training preparation
print("\n📝 Pre-training checklist:")
print("   ✓ Configuration parameters set (first cell)")
print("   ✓ All prerequisite cells executed")
print("   ✓ Data slices prepared")
print("   ✓ Sufficient memory (recommend >8GB)")

try:
    print("\n✅ Environment ready, starting training...")
    print("-" * 50)
    
    # Clear memory
    import gc
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    elif device.type == 'mps':
        torch.mps.empty_cache() if hasattr(torch.mps, 'empty_cache') else None
    
    # Record start time
    start_time = time.time()
    
    # Execute training
    td3_agent, episode_rewards = train_td3()
    
    # Calculate training time
    training_time = time.time() - start_time
    hours = int(training_time // 3600)
    minutes = int((training_time % 3600) // 60)
    seconds = int(training_time % 60)
    
    # Display results
    print("\n🎉 Training complete!")
    print(f"   Training time: {hours} hours {minutes} minutes {seconds} seconds")
    
    if episode_rewards:
        print(f"\n📊 Training statistics:")
        print(f"   Completed episodes: {len(episode_rewards)}")
        print(f"   Average reward: {np.mean(episode_rewards):.4f}")
        print(f"   Standard deviation: {np.std(episode_rewards):.4f}")
        
        # Show statistics for the last portion
        if len(episode_rewards) > 10:
            recent_n = min(50, len(episode_rewards))
            recent = episode_rewards[-recent_n:]
            print(f"\n   Last {recent_n} episodes statistics:")
            print(f"   Average: {np.mean(recent):.4f}")
            print(f"   Best: {max(recent):.4f}")
            print(f"   Worst: {min(recent):.4f}")
        
        # Overall statistics
        print(f"\n   Overall statistics:")
        print(f"   Best reward: {max(episode_rewards):.4f}")
        print(f"   Worst reward: {min(episode_rewards):.4f}")
        success = len([r for r in episode_rewards if r > 0])
        print(f"   Success rate: {success}/{len(episode_rewards)} ({success/len(episode_rewards)*100:.1f}%)")
    
    print("\n📁 Saved files:")
    print("   - Final model: checkpoints/td3/td3_model_final.pth")
    print("   - Best model: checkpoints/td3/td3_model_best.pth")
    print("   - Training statistics: logs/td3_training_stats_*.json")
    print("   - Training progress: checkpoints/td3/training_progress.json")
    
    print("\nNext steps:")
    print("1. Run the next cell for model evaluation")
    print("2. Adjust training parameters (first configuration cell)")
    print("3. View training curves and statistical analysis")
    
except KeyboardInterrupt:
    print("\n⚠️ Training interrupted by user")
    print("Tips:")
    print("- Model automatically saved to latest checkpoint")
    print("- Can resume training from checkpoint")
    
except Exception as e:
    print(f"\n❌ Error: {e}")
    print("\nSuggested solutions:")
    print("1. Check if memory is sufficient")
    print("2. Reduce BATCH_SIZE and REPLAY_BUFFER_SIZE in configuration cell")
    print("3. Reduce DATA_SAMPLE_RATE to 0.05 in configuration cell")
    print("4. Restart kernel and re-execute")
    import traceback
    traceback.print_exc()

🎯 Preparing to start TD3 training...
   Device: mps
   Data slices: 144 slices

📊 Current configuration (from global settings):
   Training episodes: 200
   Batch size: 128
   Experience buffer: 20,000
   Data sampling rate: 10%
   Max steps per episode: 500
   Save interval: Every 50 episodes

💪 Standard training mode: 200 episodes
   Expected time: 4-6 hours
   Purpose: Obtain usable strategy

💡 Tip: Adjust NUM_EPISODES in the first configuration cell

📝 Pre-training checklist:
   ✓ Configuration parameters set (first cell)
   ✓ All prerequisite cells executed
   ✓ Data slices prepared
   ✓ Sufficient memory (recommend >8GB)

✅ Environment ready, starting training...
--------------------------------------------------

🎯 Starting TD3 training
   Device: mps
   Total episodes: 200
   Random exploration steps: 500
   Batch size: 128


Training Progress:   0%|          | 0/200 [00:00<?, ?it/s]


📌 Episode 1/200

📊 Episode 50/200
   Average reward: -14.4875
   Best reward: 551.7558
   Average steps: 173
   Total steps: 8,660
   Buffer size: 8,660
   Average Critic loss: 1222.122462
   Average Actor loss: 57.099273
💾 Model saved: checkpoints/td3/td3_model_episode_50.pth

📊 Episode 100/200
   Average reward: -488.3877
   Best reward: 551.7558
   Average steps: 74
   Total steps: 12,359
   Buffer size: 12,359
   Average Critic loss: 3890.081239
   Average Actor loss: 83.544663
💾 Model saved: checkpoints/td3/td3_model_episode_100.pth

📌 Episode 101/200

📊 Episode 150/200
   Average reward: -14.4411
   Best reward: 710.4461
   Average steps: 185
   Total steps: 21,607
   Buffer size: 20,000
   Average Critic loss: 2712.140176
   Average Actor loss: 55.104702
💾 Model saved: checkpoints/td3/td3_model_episode_150.pth

📊 Episode 200/200
   Average reward: 79.6802
   Best reward: 710.4461
   Average steps: 184
   Total steps: 30,829
   Buffer size: 20,000
   Average Critic loss: 1822.93

In [10]:
# Cell 9: Model Evaluation

def evaluate_td3_agent(td3_agent, env_manager, num_eval_episodes=10):
    """Evaluate trained TD3 agent"""
    
    print(f"\n🔍 Evaluating TD3 agent performance...")
    print(f"   Evaluation episodes: {num_eval_episodes}")
    
    eval_rewards = []
    eval_actions = []
    eval_pnls = []
    
    for episode in tqdm(range(num_eval_episodes), desc="Evaluation Progress"):
        # Get evaluation environment
        env, data_file = env_manager.get_random_env()
        
        # Reset environment
        tensordict = env.reset()
        state = tensordict['observation'].cpu().numpy()
        
        episode_reward = 0
        episode_actions = []
        done = False
        steps = 0
        
        while not done and steps < TD3_CONFIG['max_steps_per_episode']:
            # Use deterministic policy (evaluation mode, no noise added)
            action = td3_agent.select_action(state, add_noise=False)
            episode_actions.append(action.tolist())
            
            # Execute action
            action_tensor = torch.tensor(action, dtype=torch.float32, device=device)
            action_td = TensorDict({"action": action_tensor}, batch_size=(), device=device)
            next_tensordict = env.step(action_td)
            
            if 'next' in next_tensordict:
                state = next_tensordict['next']['observation'].cpu().numpy()
                reward = next_tensordict['next']['reward'].cpu().item()
                done = next_tensordict['next']['done'].cpu().item() > 0.5
                
                episode_reward += reward
                steps += 1
                
                # Record final PnL
                if done or steps >= TD3_CONFIG['max_steps_per_episode'] - 1:
                    strategy_state = env._get_strategy_state()
                    eval_pnls.append(strategy_state['pnl'])
            else:
                done = True
        
        eval_rewards.append(episode_reward)
        eval_actions.append(episode_actions)
        
        # Clean up environment
        del env
    
    # Statistical analysis
    avg_reward = np.mean(eval_rewards)
    std_reward = np.std(eval_rewards)
    avg_pnl = np.mean(eval_pnls)
    success_rate = len([r for r in eval_rewards if r > 0]) / len(eval_rewards) * 100
    
    print(f"\n📊 Evaluation results:")
    print(f"   Average reward: {avg_reward:.4f} ± {std_reward:.4f}")
    print(f"   Average PnL: ${avg_pnl:.2f}")
    print(f"   Success rate: {success_rate:.1f}%")
    print(f"   Best reward: {max(eval_rewards):.4f}")
    print(f"   Worst reward: {min(eval_rewards):.4f}")
    
    # Analyze learned strategy parameters
    if eval_actions:
        all_actions = [action for episode in eval_actions for action in episode]
        actions_array = np.array(all_actions)
        
        avg_params = np.mean(actions_array, axis=0)
        std_params = np.std(actions_array, axis=0)
        
        print(f"\n🎯 Learned strategy parameters:")
        print(f"   {'Parameter':<15} {'Average':<15} {'Std Dev':<10}")
        print(f"   {'-'*40}")
        
        param_names = ['Half Spread (ticks)', 'Skew Factor (ticks)', 'Grid Layers', 'Grid Interval (ticks)']
        for i, name in enumerate(param_names):
            print(f"   {name:<15} {avg_params[i]:>6.1f} ± {std_params[i]:<8.1f}")
        
        print(f"\n   📌 Actual usage parameters (after rounding):")
        actual_params = np.round(avg_params).astype(int)
        print(f"   Half spread: {actual_params[0]} ticks")
        print(f"   Skew factor: {actual_params[1]} ticks")
        print(f"   Grid layers: {actual_params[2]} layers")
        print(f"   Grid interval: {actual_params[3]} ticks")
    
    return {
        'rewards': eval_rewards,
        'pnls': eval_pnls,
        'actions': eval_actions,
        'avg_reward': avg_reward,
        'avg_pnl': avg_pnl,
        'success_rate': success_rate
    }


# If training is complete, perform evaluation
if 'td3_agent' in locals():
    eval_results = evaluate_td3_agent(td3_agent, env_manager, num_eval_episodes=10)
    
    # Save evaluation results
    eval_path = f"logs/td3_eval_results_{time.strftime('%Y%m%d_%H%M%S')}.json"
    with open(eval_path, 'w') as f:
        json.dump({
            'avg_reward': eval_results['avg_reward'],
            'avg_pnl': eval_results['avg_pnl'],
            'success_rate': eval_results['success_rate'],
            'rewards': eval_results['rewards'],
            'pnls': eval_results['pnls']
        }, f, indent=2)
    print(f"\n💾 Evaluation results saved: {eval_path}")
else:
    print("⚠️ Please run the training cell first")


🔍 Evaluating TD3 agent performance...
   Evaluation episodes: 10


Evaluation Progress:   0%|          | 0/10 [00:00<?, ?it/s]


📊 Evaluation results:
   Average reward: -8.2936 ± 143.0671
   Average PnL: $-8.27
   Success rate: 60.0%
   Best reward: 150.4571
   Worst reward: -248.8367

🎯 Learned strategy parameters:
   Parameter       Average         Std Dev   
   ----------------------------------------
   Half Spread (ticks)   19.7 ± 2.2     
   Skew Factor (ticks)   16.6 ± 14.4    
   Grid Layers        5.0 ± 0.0     
   Grid Interval (ticks)   17.5 ± 3.2     

   📌 Actual usage parameters (after rounding):
   Half spread: 20 ticks
   Skew factor: 17 ticks
   Grid layers: 5 layers
   Grid interval: 17 ticks

💾 Evaluation results saved: logs/td3_eval_results_20250812_084405.json


## 📊 Summary

This notebook successfully implements a PMM market making strategy reinforcement learning training system based on the **TD3 (Twin Delayed Deep Deterministic Policy Gradient)** algorithm.

### ✅ Core Features of TD3

1. **Deterministic Policy**: Directly outputs actions with high computational efficiency
2. **Delayed Policy Updates**: Updates Actor every 2 steps to improve stability
3. **Target Policy Smoothing**: Adds noise to target actions to prevent overfitting
4. **Twin Q-Networks**: Uses the smaller Q-value as target to reduce overestimation

### 🎯 TD3 vs SAC Comparison

| Aspect | TD3 | SAC |
|------|-----|-----|
| **Policy Type** | Deterministic | Stochastic |
| **Exploration Mechanism** | Gaussian Noise | Entropy Regularization |
| **Computational Complexity** | Lower | Higher |
| **Sample Efficiency** | Higher | Medium |
| **Stability** | Very Good | Good |
| **Use Cases** | Smaller Action Spaces | Tasks Requiring Exploration |

### 🚀 Usage Recommendations

1. **Hyperparameter Tuning**
   - `exploration_noise`: Controls exploration degree (0.05-0.2)
   - `policy_noise`: Target smoothing noise (0.1-0.3)
   - `policy_freq`: Policy update frequency (2-4)

2. **Training Tips**
   - Start with larger exploration noise
   - Gradually reduce noise to refine policy
   - Monitor balance between Actor and Critic losses

3. **Performance Optimization**
   - TD3 typically trains faster than SAC
   - Suitable for high-frequency trading requiring quick decisions
   - Performs better in deterministic environments

### 💡 Future Improvements

1. **Algorithm Enhancements**
   - Implement TD3+BC (Behavior Cloning)
   - Add prioritized experience replay
   - Integrate imitation learning

2. **Strategy Improvements**
   - Adaptive noise adjustment
   - Multi-timescale training
   - Meta-learning for fast adaptation

3. **Practical Applications**
   - Real-time trading system integration
   - Multi-asset portfolio optimization
   - Enhanced risk management